In [ ]:
%pip install gradio
%pip install openai
%pip install tiktoken

In [10]:
import os

# ホームディレクトリの下にあるファイルのパスを作成
file_path = os.path.join(os.path.expanduser("~"), 'dotsecret')

# ファイルを開いて各行を読みます
variables = {}
with open(file_path, 'r') as file:
    for line in file:
        # 空白行を無視します
        if line.strip():
            key, value = line.strip().split('=')
            os.environ[key] = value

In [2]:
import openai
import random
import gradio as gr
import os

# OpenAI APIキーを設定
openai.api_key = os.getenv("OPENAI_API_KEY")

def openai_chat(message, history):
    return random.choice(["Yes", "No"])

with gr.Blocks() as app:
    with gr.Tab("Chat"):
        gr.ChatInterface(fn=openai_chat)

if __name__ == "__main__":
    app.launch()


Running on local URL:  http://127.0.0.1:7877

To create a public link, set `share=True` in `launch()`.


In [32]:
import tiktoken
import openai
import gradio as gr
import os

# OpenAI APIキーを設定
openai.api_key = os.getenv("OPENAI_API_KEY")

# Get the encoding object
enc = tiktoken.get_encoding("cl100k_base")

def calculate_tokens(text):
    tokens = enc.encode(text)
    return len(tokens)

def truncate_history(history, max_tokens=4096):
    total_tokens = 0
    truncated_history = []
    # Start iterating from the newest message
    for message in reversed(history):
        # +4 for the tokens associated with role and content keys and their values
        message_tokens = calculate_tokens(message['content']) + calculate_tokens(f"{message['role']}: {message['content']}")
        if total_tokens + message_tokens > max_tokens:
            break  # Stop if adding the next message would exceed the max tokens
        total_tokens += message_tokens
        truncated_history.append(message)
    return list(reversed(truncated_history))  # Reverse again to maintain original order

def openai_chat(messages, history):
    formatted_messages = [{"role": "system", "content": "You are a helpful assistant."}]

    # Convert the history list to the required message format
    for inner_list in history:
        for i, content in enumerate(inner_list):
            role = "user" if i % 2 == 0 else "assistant"  # Assume user's messages are at even indices, and assistant's messages are at odd indices
            formatted_messages.append({"role": role, "content": content})
    
    # Add the new message from the user
    formatted_messages.append({"role": "user", "content": messages})
    
    # Truncate the history if necessary
    formatted_messages = truncate_history(formatted_messages)
    
    # OpenAI API call
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=formatted_messages
    )
    reply = response['choices'][0]['message']['content']
    return reply

with gr.Blocks() as app:
    with gr.Tab("Chat"):
        gr.ChatInterface(fn=openai_chat)

if __name__ == "__main__":
    app.launch()


Running on local URL:  http://127.0.0.1:7903

To create a public link, set `share=True` in `launch()`.
